# **Naira to Dollar Exchange Rate Forecast**

In [ ]:
# import required pacakges
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping    



In [ ]:
# import lstm


In [2]:
# load features
df_feat = pd.read_csv('../data/03_features/ngn_us_exchange_rates_features.csv')

# Display the feature engineered data
df_feat.head()

,Date,Rate,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,rmean_7,rstd_7,...,ret_7,ret_14,ret_21,ret_30,dayofweek,month,day,is_month_start,is_month_end,year
0,1995-08-10,83.0,83.5,84.0,84.0,83.8,84.3,81.5,83.814286,0.167616,...,-0.009547,-0.015421,0.012195,0.018405,3,8,10,0,0,1995
1,1995-08-11,83.0,83.0,83.5,84.0,83.8,84.5,82.0,83.700000,0.351188,...,-0.009547,-0.017751,0.006061,0.012195,4,8,11,0,0,1995
2,1995-08-12,83.0,83.0,83.0,83.5,83.8,84.5,81.5,83.585714,0.433699,...,-0.009547,-0.017751,0.006061,0.018405,5,8,12,0,0,1995
3,1995-08-13,83.0,83.0,83.0,83.0,83.8,84.5,82.5,83.471429,0.471573,...,-0.009547,-0.017751,0.006061,0.006061,6,8,13,0,0,1995
4,1995-08-14,84.1,83.0,83.0,83.0,84.0,83.5,82.5,83.357143,0.475595,...,0.001190,0.007186,0.019394,0.019394,0,8,14,0,0,1995


In [5]:
# Convert transaction_time column to date format
df_feat["Date"] = pd.to_datetime(df_feat["Date"], errors="coerce")

In [7]:
# Train-test split (time split)
if len(df_feat) > 400:
    test_size = 180
else:
    test_size = int(len(df_feat)*0.2)
train = df_feat.iloc[:-test_size].copy()
test = df_feat.iloc[-test_size:].copy()

print("Train range:", train['Date'].min().date(), "to", train['Date'].max().date())
print("Test range:", test['Date'].min().date(), "to", test['Date'].max().date())

Train range: 1995-08-10 to 2025-04-26
Test range: 2025-04-27 to 2025-10-23


In [8]:
# Features and target
feature_cols = [c for c in df_feat.columns if c not in ['Date','Rate']]
X_train = train[feature_cols]
y_train = train['Rate']
X_test = test[feature_cols]
y_test = test['Rate']

In [10]:
# Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
# Metrics storage
results = []

In [15]:
# Baseline 1: Naive (y_hat = last observed rate)
y_pred_naive = np.repeat(train['Rate'].iloc[-1], len(y_test))
mae_naive = mean_absolute_error(y_test, y_pred_naive)
rmse_naive = mean_squared_error(y_test, y_pred_naive)
mape_naive = np.mean(np.abs((y_test - y_pred_naive) / y_test)) * 100
results.append(("NaiveLastValue", mae_naive, rmse_naive, mape_naive))

In [17]:
# Baseline 2: Persistence (y_hat = yesterday's rate) -> use lag_1 from test
y_pred_pers = X_test['lag_1'].values
mae_pers = mean_absolute_error(y_test, y_pred_pers)
rmse_pers = mean_squared_error(y_test, y_pred_pers)
mape_pers = np.mean(np.abs((y_test - y_pred_pers) / y_test)) * 100
results.append(("PersistenceLag1", mae_pers, rmse_pers, mape_pers))

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(("RandomForest", mean_absolute_error(y_test, y_pred_rf),
                mean_squared_error(y_test, y_pred_rf, squared=False),
                np.mean(np.abs((y_test - y_pred_rf) / y_test)) * 100))